In [1]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader



In [2]:

dir_loader = DirectoryLoader(
    path="./pdf_documents",  # current directory (AGENTICRAG)
    glob="Residence_Certificate*.pdf",  # only your two PDFs
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

pdf_documents = dir_loader.load()

pdf_documents


100%|██████████| 2/2 [00:01<00:00,  1.48it/s]


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-01-25T06:57:18+00:00', 'source': 'pdf_documents\\Residence_Certificate_Full_Structured_Info.pdf', 'file_path': 'pdf_documents\\Residence_Certificate_Full_Structured_Info.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-01-25T06:57:18+00:00', 'trapped': '', 'modDate': "D:20260125065718+00'00'", 'creationDate': "D:20260125065718+00'00'", 'page': 0}, page_content='Residence Certificate – Structured Information\nService Details\nService ID\n75\nDepartment\nRevenue Administration\nService Name\nResidence Certificate\nAccess Type\nOperator\nOnline Availability\nYes\nService Charge (INR)\n60\nThis document contains cleanly structured and standardized information related to the Residence\nCertificate service under Revenue Administration. The tabular service details

In [3]:
import sys
print(sys.executable)


e:\DESKTOP\ORCHESTRA\AGENTICRAG\.venv\Scripts\python.exe


In [4]:
# for doc in chunks:
#     source = doc.metadata.get("source", "").lower()

#     if "reason" in source or "explanation" in source:
#         doc.metadata["intent"] = "reasoning"
#     else:
#         doc.metadata["intent"] = "rules"

#     doc.metadata["service"] = "residence_certificate"


In [5]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_files = {
    "rules": "pdf_documents/Residence_Certificate_Full_Structured_Info.pdf",
    "reasoning": "pdf_documents/Residence_Certificate_Service_Explanation_and_Document_Reasons.pdf"
}


documents = {}

for key, path in pdf_files.items():
    loader = PyMuPDFLoader(path)
    documents[key] = loader.load()


In [6]:
from langchain_core.documents import Document

def make_chunk(text, intent, source, entity=None):
    metadata = {
        "service": "residence_certificate",
        "intent": intent,
        "source": source
    }
    if entity:
        metadata["entity"] = entity

    return Document(page_content=text.strip(), metadata=metadata)



In [7]:
#Semantic chunking for PDF 1

rule_chunks = []

rules_text = "\n".join([d.page_content for d in documents["rules"]])

# --- Service Info ---
service_info = rules_text.split("Mandatory Documents")[0]
rule_chunks.append(
    make_chunk(service_info, "service_info", pdf_files["rules"])
)

# --- Mandatory Documents ---
mandatory_section = rules_text.split("Mandatory Documents")[1].split("Category-wise")[0]
rule_chunks.append(
    make_chunk(mandatory_section, "rules_mandatory", pdf_files["rules"])
)

# --- Category-wise Sections ---
citizen_section = rules_text.split("General Citizens")[1].split("Government employees")[0]
rule_chunks.append(
    make_chunk(citizen_section, "rules_category_citizen", pdf_files["rules"])
)

govt_section = rules_text.split("Government employees")[1]
rule_chunks.append(
    make_chunk(govt_section, "rules_category_govt", pdf_files["rules"])
)


In [8]:
#Semantic chunking for PDF 2

reasoning_chunks = []

reasoning_text = "\n".join([d.page_content for d in documents["reasoning"]])

# --- Service explanation ---
service_explanation = reasoning_text.split("Why Documents are Collected")[0]
reasoning_chunks.append(
    make_chunk(service_explanation, "service_explanation", pdf_files["reasoning"])
)

# --- Global reasoning ---
global_reason = reasoning_text.split("Why Documents are Collected")[1].split("Reasoning Behind")[0]
reasoning_chunks.append(
    make_chunk(global_reason, "reasoning_global", pdf_files["reasoning"])
)

# --- Per-document reasoning ---
document_reasons = {
    "Applicant Photograph": "Applicant Photograph",
    "Current Address Proof": "Current Address Proof",
    "Self-Declaration": "Self-Declaration",
    "Passport": "Passport",
    "Driving Licence": "Driving Licence",
    "PAN Card": "PAN Card",
    "Bank / Post Office Passbook": "Bank / Post Office Passbook",
    "Smart Card": "Smart Card",
    "Health Insurance Smart Card": "Health Insurance Smart Card",
    "Pension Document": "Pension Document",
    "Service Identity Card": "Service Identity Card",
    "MP/MLA/MLC Identity Card": "MP/MLA/MLC Identity Card",
    "Photo Voter Slip": "Photo Voter Slip"
}

for doc_name, key in document_reasons.items():
    if doc_name in reasoning_text:
        section = reasoning_text.split(doc_name)[1].split("\n", 1)[1]
        reasoning_chunks.append(
            make_chunk(section, "reasoning_document", pdf_files["reasoning"], entity=doc_name)
        )


In [9]:
all_chunks = rule_chunks + reasoning_chunks

print(f"Total semantic chunks created: {len(all_chunks)}")

# Inspect one
print(all_chunks[0].metadata)
print(all_chunks[0].page_content[:300])


Total semantic chunks created: 19
{'service': 'residence_certificate', 'intent': 'service_info', 'source': 'pdf_documents/Residence_Certificate_Full_Structured_Info.pdf'}
Residence Certificate – Structured Information
Service Details
Service ID
75
Department
Revenue Administration
Service Name
Residence Certificate
Access Type
Operator
Online Availability
Yes
Service Charge (INR)
60
This document contains cleanly structured and standardized information related to the


In [10]:
from dotenv import load_dotenv
import os

load_dotenv()  # 👈 this reads .env into environment variables

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY not found")



In [11]:
from openai import OpenAI

client_embed = OpenAI(
    api_key=NVIDIA_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1"
)


In [12]:
def embed_chunks(chunks):
    """
    chunks: List[langchain_core.documents.Document]
    returns: List[List[float]] -> embeddings aligned with chunks
    """

    texts = [chunk.page_content for chunk in chunks]

    response = client_embed.embeddings.create(
        model="nvidia/nv-embedqa-e5-v5",
        input=texts,
        extra_body={
            "input_type": "passage"  # REQUIRED for document chunks
        }
    )

    return [item.embedding for item in response.data]


In [13]:
# %% 
chunk_embeddings = embed_chunks(all_chunks)

# sanity check
print(len(chunk_embeddings))       # should be 19
print(len(chunk_embeddings[0]))    # embedding dimension (~1024)


19
1024


In [14]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url="https://103795bc-13b7-45b8-aea6-9b2ab07095a1.eu-west-2-0.aws.cloud.qdrant.io", 
    api_key=QDRANT_API_KEY,
)



In [15]:
from qdrant_client.models import VectorParams, Distance

VECTOR_SIZE = len(chunk_embeddings[0])

COLLECTION_NAME = "residence_certificate_agent"

qdrant_client.recreate_collection(
    collection_name="residence_certificate_agent",
    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE   # ✅ BEST for E5 models
    )
)


C:\Users\deva\AppData\Local\Temp\ipykernel_20404\511622667.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


ResponseHandlingException: [Errno 11001] getaddrinfo failed

In [ ]:
qdrant_client.get_collections()
